# 02 — Deepfake KYC: Model Development

**Goal:** Train and evaluate deepfake detection models for the RISKNET backend.

**Datasets:** FaceForensics++ and Celeb-DF v2

**Outputs:** `weights/efficientnet_best.pt` `weights/vit_best.pt` `weights/frequency_cnn_best.pt` `precomputed/kyc_metrics.json`

**Research path:** Simple CNN -> EfficientNet-B4 -> ViT-B/16 -> FrequencyCNN -> Ensemble

## 0. Setup

In [ ]:
# Run once to install dependencies
# !pip install torch torchvision timm facenet-pytorch opencv-python-headless scikit-learn scipy pandas numpy Pillow

In [ ]:
import os, json, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from facenet_pytorch import MTCNN
from scipy.fft import dctn
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, precision_recall_curve,
)
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")
print("PyTorch:", torch.__version__)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
# Backend-compatible constants — do not change
KYC_MAX_VIDEO_FRAMES = 5
KYC_FRAME_SIZE = 224

ROOT            = Path("../../")          # repo root
DATA_DIR        = ROOT / "data" / "kyc"
WEIGHTS_DIR     = ROOT / "backend" / "weights"
PRECOMPUTED_DIR = ROOT / "backend" / "precomputed"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
PRECOMPUTED_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 1. Dataset Preparation

| Dataset | ~Real | ~Fake | Purpose |
|---|---|---|---|
| FaceForensics++ | 1 000 videos | 4 000 videos | Train / eval |
| Celeb-DF v2 | 590 videos | 5 639 videos | Train / eval |

Place downloaded videos under `data/kyc/raw/faceforensics/` and `data/kyc/raw/celebdf/`, each with `real/` and `fake/` sub-folders containing `.mp4` files.

In [ ]:
RAW_DIR = DATA_DIR / "raw"
sources = {
    "faceforensics": RAW_DIR / "faceforensics",
    "celebdf":       RAW_DIR / "celebdf",
}
for name, path in sources.items():
    real_n = len(list((path/"real").glob("*.mp4"))) if (path/"real").exists() else 0
    fake_n = len(list((path/"fake").glob("*.mp4"))) if (path/"fake").exists() else 0
    print(f"{name:20s}  real={real_n:>5}  fake={fake_n:>5}")

## 2. Common Preprocessing Pipeline

In [ ]:
def extract_frames(video_path, n=KYC_MAX_VIDEO_FRAMES):
    """Sample n uniformly-spaced frames. Returns list of PIL Images."""
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total == 0:
        cap.release(); return []
    indices = [int(i * total / n) for i in range(n)]
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
    cap.release()
    return frames

In [ ]:
# Same MTCNN config as the backend: keep_all=False -> largest face only
_mtcnn = MTCNN(
    image_size=KYC_FRAME_SIZE, margin=20,
    keep_all=False, device=DEVICE, post_process=False,
)

def detect_face(image):
    """Return 224x224 PIL face crop, or None."""
    tensor = _mtcnn(image)
    if tensor is None: return None
    arr = tensor.permute(1, 2, 0).numpy().clip(0, 255).astype("uint8")
    return Image.fromarray(arr).resize((KYC_FRAME_SIZE, KYC_FRAME_SIZE))

In [ ]:
CROPS_DIR = DATA_DIR / "crops"
(CROPS_DIR / "real").mkdir(parents=True, exist_ok=True)
(CROPS_DIR / "fake").mkdir(parents=True, exist_ok=True)

def process_videos(source_name, label):
    video_dir = sources[source_name] / label
    if not video_dir.exists():
        print(f"Skipping {source_name}/{label} — not found"); return
    videos = list(video_dir.glob("*.mp4"))
    out_dir = CROPS_DIR / label
    saved = 0
    for vid in videos:
        for fi, frame in enumerate(extract_frames(vid)):
            face = detect_face(frame)
            if face is None: continue
            face.save(out_dir / f"{source_name}_{vid.stem}_f{fi}.jpg", quality=92)
            saved += 1
    print(f"{source_name}/{label}: {len(videos)} videos -> {saved} face crops")

for src in sources:
    process_videos(src, "real")
    process_videos(src, "fake")

In [ ]:
# Video-level 80/10/10 split so frames from the same video stay in one split
def build_split(label):
    crops = list((CROPS_DIR / label).glob("*.jpg"))
    groups = {}
    for p in crops:
        key = p.stem.rsplit("_", 1)[0]   # drop _f<N> suffix
        groups.setdefault(key, []).append(p)
    videos = list(groups.keys())
    train_v, temp_v = train_test_split(videos, test_size=0.2, random_state=SEED)
    val_v, test_v   = train_test_split(temp_v, test_size=0.5, random_state=SEED)
    return {s: [p for v in vs for p in groups[v]]
            for s, vs in [("train",train_v),("val",val_v),("test",test_v)]}

real_splits = build_split("real")
fake_splits = build_split("fake")

split_dfs = {}
for split in ("train", "val", "test"):
    rows = [(str(p), 0) for p in real_splits[split]] + \
           [(str(p), 1) for p in fake_splits[split]]
    split_dfs[split] = pd.DataFrame(rows, columns=["path", "label"])
    n = len(split_dfs[split])
    nr = (split_dfs[split].label == 0).sum()
    nf = (split_dfs[split].label == 1).sum()
    print(f"{split:5s}: {n} samples  (real={nr}, fake={nf})")

## 3. Evaluation Framework

In [ ]:
def evaluate(y_true, y_pred_prob, threshold=0.5, title="Model"):
    """6 metrics + confusion matrix + ROC + PR curves."""
    y_pred = (np.array(y_pred_prob) >= threshold).astype(int)
    y_true = np.array(y_true)
    metrics = {
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall":    recall_score(y_true, y_pred, zero_division=0),
        "F1":        f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC":   roc_auc_score(y_true, y_pred_prob),
        "PR-AUC":    average_precision_score(y_true, y_pred_prob),
    }
    print(f"\n--- {title} ---")
    for k, v in metrics.items(): print(f"  {k:12s}: {v:.4f}")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(title)
    cm = confusion_matrix(y_true, y_pred)
    axes[0].imshow(cm, cmap="Blues")
    axes[0].set_title("Confusion Matrix")
    axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
    for i in range(2):
        for j in range(2):
            axes[0].text(j, i, cm[i,j], ha="center", va="center", fontsize=14)
    axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
    axes[0].set_xticklabels(["Real","Fake"]); axes[0].set_yticklabels(["Real","Fake"])
    fpr, tpr, _ = roc_curve(y_true, y_pred_prob)
    axes[1].plot(fpr, tpr, lw=2, label=f"AUC={metrics['ROC-AUC']:.3f}")
    axes[1].plot([0,1],[0,1],"--",color="grey")
    axes[1].set_title("ROC Curve"); axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR")
    axes[1].legend()
    prec, rec, _ = precision_recall_curve(y_true, y_pred_prob)
    axes[2].plot(rec, prec, lw=2, label=f"AP={metrics['PR-AUC']:.3f}")
    axes[2].set_title("PR Curve"); axes[2].set_xlabel("Recall"); axes[2].set_ylabel("Precision")
    axes[2].legend()
    plt.tight_layout(); plt.show()
    return metrics

In [ ]:
def threshold_analysis(y_true, y_pred_prob, title="Threshold Analysis"):
    thresholds = np.linspace(0.1, 0.9, 50)
    f1s, precs, recs = [], [], []
    for t in thresholds:
        yp = (np.array(y_pred_prob) >= t).astype(int)
        f1s.append(f1_score(y_true, yp, zero_division=0))
        precs.append(precision_score(y_true, yp, zero_division=0))
        recs.append(recall_score(y_true, yp, zero_division=0))
    plt.figure(figsize=(8,4))
    plt.plot(thresholds, f1s,   label="F1")
    plt.plot(thresholds, precs, label="Precision")
    plt.plot(thresholds, recs,  label="Recall")
    plt.axvline(0.50, color="red",    linestyle="--", label="suspicious (0.50)")
    plt.axvline(0.75, color="orange", linestyle="--", label="high_risk (0.75)")
    plt.title(title); plt.xlabel("Threshold"); plt.legend()
    plt.tight_layout(); plt.show()

## 4. DataLoader Helpers

Shared by all RGB experiments. FrequencyCNN uses a DCT-specific loader below.

In [ ]:
IMAGENET_TRANSFORM = transforms.Compose([
    transforms.Resize((KYC_FRAME_SIZE, KYC_FRAME_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])
AUGMENT_TRANSFORM = transforms.Compose([
    transforms.Resize((KYC_FRAME_SIZE, KYC_FRAME_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

class FaceDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform or IMAGENET_TRANSFORM
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return self.transform(Image.open(row["path"]).convert("RGB")), int(row["label"])

def make_loaders(batch_size=32):
    train_dl = DataLoader(FaceDataset(split_dfs["train"], AUGMENT_TRANSFORM),
                          batch_size=batch_size, shuffle=True,  num_workers=2)
    val_dl   = DataLoader(FaceDataset(split_dfs["val"]),
                          batch_size=batch_size, shuffle=False, num_workers=2)
    test_dl  = DataLoader(FaceDataset(split_dfs["test"]),
                          batch_size=batch_size, shuffle=False, num_workers=2)
    return train_dl, val_dl, test_dl

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train(); total_loss = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)

def evaluate_loader(model, loader):
    """Return (labels, fake_probs) for a full DataLoader."""
    model.eval(); all_labels, all_probs = [], []
    with torch.no_grad():
        for x, y in loader:
            prob = F.softmax(model(x.to(DEVICE)), dim=1)[:,1].cpu().numpy()
            all_probs.extend(prob); all_labels.extend(y.numpy())
    return np.array(all_labels), np.array(all_probs)

def train_model(model, train_dl, val_dl, optimizer, scheduler,
                epochs=20, save_path=None, patience=5):
    """Train with early stopping on val ROC-AUC."""
    criterion = nn.CrossEntropyLoss()
    best_auc, wait = 0.0, 0
    history = {"train_loss": [], "val_auc": []}
    for epoch in range(1, epochs + 1):
        loss = train_one_epoch(model, train_dl, optimizer, criterion)
        labels, probs = evaluate_loader(model, val_dl)
        val_auc = roc_auc_score(labels, probs)
        history["train_loss"].append(loss)
        history["val_auc"].append(val_auc)
        if scheduler: scheduler.step()
        print(f"Epoch {epoch:02d}/{epochs}  loss={loss:.4f}  val_auc={val_auc:.4f}")
        if val_auc > best_auc:
            best_auc, wait = val_auc, 0
            if save_path:
                torch.save(model.state_dict(), save_path)
                print(f"  -> Checkpoint saved (auc={best_auc:.4f})")
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping at epoch {epoch}"); break
    return history

In [ ]:
def show_examples(indices, probs, title, n=6):
    """Display face crops with their model scores."""
    indices = list(indices[:n])
    if not indices: print(f"{title}: none"); return
    fig, axes = plt.subplots(1, len(indices), figsize=(3*len(indices), 3))
    if len(indices) == 1: axes = [axes]
    for ax, i in zip(axes, indices):
        ax.imshow(Image.open(test_paths[i])); ax.axis("off")
        ax.set_title(f"p={probs[i]:.2f}", fontsize=8)
    fig.suptitle(title); plt.tight_layout(); plt.show()

---
## EXPERIMENT 1 — Simple CNN Baseline

**Question:** How far can a small spatial CNN get without pretrained weights?

> Baseline only. No backend impact.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),nn.ReLU(), nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*4*4, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )
    def forward(self, x): return self.classifier(self.features(x))

In [ ]:
train_dl, val_dl, test_dl = make_loaders(batch_size=32)
test_paths = split_dfs["test"]["path"].values  # reused in later cells

simple_cnn = SimpleCNN().to(DEVICE)
opt_cnn = torch.optim.Adam(simple_cnn.parameters(), lr=1e-3)

history_cnn = train_model(
    simple_cnn, train_dl, val_dl,
    optimizer=opt_cnn, scheduler=None,
    epochs=15, save_path=None,   # baseline — no checkpoint
)

In [ ]:
labels_cnn, probs_cnn = evaluate_loader(simple_cnn, test_dl)
metrics_cnn = evaluate(labels_cnn, probs_cnn, title="Simple CNN")

results_table = pd.DataFrame([{"Model": "Simple CNN",
    **{k: round(v,4) for k,v in metrics_cnn.items()}}])
results_table

---
## EXPERIMENT 2 — EfficientNet-B4

**Question:** Does a strong pretrained CNN improve detection?

**Backend impact:** saves `weights/efficientnet_best.pt`

In [ ]:
effnet = timm.create_model("efficientnet_b4", pretrained=True, num_classes=2)

# Freeze all, then unfreeze last 2 blocks + classifier
for p in effnet.parameters(): p.requires_grad = False
for block in list(effnet.blocks.children())[-2:]:
    for p in block.parameters(): p.requires_grad = True
for p in effnet.conv_head.parameters(): p.requires_grad = True
for p in effnet.classifier.parameters(): p.requires_grad = True

effnet = effnet.to(DEVICE)
print(f"Trainable params: {sum(p.numel() for p in effnet.parameters() if p.requires_grad):,}")

In [ ]:
opt_eff   = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, effnet.parameters()), lr=1e-4, weight_decay=1e-2)
sched_eff = torch.optim.lr_scheduler.CosineAnnealingLR(opt_eff, T_max=20)
EFFNET_CKPT = WEIGHTS_DIR / "efficientnet_best.pt"

history_eff = train_model(
    effnet, train_dl, val_dl,
    optimizer=opt_eff, scheduler=sched_eff,
    epochs=20, save_path=EFFNET_CKPT, patience=5
)

In [ ]:
effnet.load_state_dict(torch.load(EFFNET_CKPT, map_location=DEVICE))
labels_eff, probs_eff = evaluate_loader(effnet, test_dl)
metrics_eff = evaluate(labels_eff, probs_eff, title="EfficientNet-B4")

In [ ]:
# Inspect hard examples
preds_eff = (probs_eff >= 0.5).astype(int)
fp_idx = np.where((labels_eff == 0) & (preds_eff == 1))[0]  # Real -> Fake
fn_idx = np.where((labels_eff == 1) & (preds_eff == 0))[0]  # Fake -> Real
show_examples(fp_idx, probs_eff, "EfficientNet FP — Real predicted as Fake")
show_examples(fn_idx, probs_eff, "EfficientNet FN — Fake predicted as Real")

In [ ]:
results_table = pd.concat([results_table, pd.DataFrame([{"Model": "EfficientNet-B4",
    **{k: round(v,4) for k,v in metrics_eff.items()}}])], ignore_index=True)
results_table

---
## EXPERIMENT 3 — ViT-B/16

**Question:** Does a transformer representation capture complementary evidence?

**Backend impact:** saves `weights/vit_best.pt`

In [ ]:
vit = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=2)

# Freeze all, then unfreeze last 4 blocks + norm + head
for p in vit.parameters(): p.requires_grad = False
for block in vit.blocks[-4:]:
    for p in block.parameters(): p.requires_grad = True
for p in vit.norm.parameters(): p.requires_grad = True
for p in vit.head.parameters(): p.requires_grad = True

vit = vit.to(DEVICE)
print(f"Trainable params: {sum(p.numel() for p in vit.parameters() if p.requires_grad):,}")

In [ ]:
opt_vit   = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, vit.parameters()), lr=5e-5, weight_decay=1e-2)
sched_vit = torch.optim.lr_scheduler.CosineAnnealingLR(opt_vit, T_max=20)
VIT_CKPT  = WEIGHTS_DIR / "vit_best.pt"

history_vit = train_model(
    vit, train_dl, val_dl,
    optimizer=opt_vit, scheduler=sched_vit,
    epochs=20, save_path=VIT_CKPT, patience=5
)

In [ ]:
vit.load_state_dict(torch.load(VIT_CKPT, map_location=DEVICE))
labels_vit, probs_vit = evaluate_loader(vit, test_dl)
metrics_vit = evaluate(labels_vit, probs_vit, title="ViT-B/16")

In [ ]:
# Do EfficientNet and ViT make different mistakes?
preds_vit = (probs_vit >= 0.5).astype(int)
eff_ok = preds_eff == labels_eff
vit_ok = preds_vit == labels_vit

print("Error overlap: EfficientNet vs ViT")
print(f"  Both correct      : {(eff_ok & vit_ok).sum()}")
print(f"  EfficientNet only : {(eff_ok & ~vit_ok).sum()}")
print(f"  ViT only          : {(~eff_ok & vit_ok).sum()}")
print(f"  Both wrong        : {(~eff_ok & ~vit_ok).sum()}")

In [ ]:
results_table = pd.concat([results_table, pd.DataFrame([{"Model": "ViT-B/16",
    **{k: round(v,4) for k,v in metrics_vit.items()}}])], ignore_index=True)
results_table

---
## EXPERIMENT 4 — Frequency-Domain CNN (DCT)

**Question:** Can frequency artifacts reveal what RGB models miss?

**Backend impact:** saves `weights/frequency_cnn_best.pt`

In [ ]:
# Matches backend/services/kyc/frequency_analyzer.extract_dct_features exactly
def extract_dct_features(face_image, size=224):
    gray    = face_image.convert("L").resize((size, size))
    arr     = np.array(gray, dtype=np.float32) / 255.0
    dct     = dctn(arr, norm="ortho")
    log_mag = np.log1p(np.abs(dct))
    log_mag = (log_mag - log_mag.min()) / (log_mag.max() - log_mag.min() + 1e-8)
    return log_mag[np.newaxis, :, :]   # shape: (1, H, W)

In [ ]:
# Visualize DCT maps: real vs fake
sample_real = split_dfs["test"][split_dfs["test"]["label"]==0].sample(3, random_state=SEED)
sample_fake = split_dfs["test"][split_dfs["test"]["label"]==1].sample(3, random_state=SEED)

fig, axes = plt.subplots(2, 3, figsize=(10, 6))
for col, (_, row) in enumerate(sample_real.iterrows()):
    dct = extract_dct_features(Image.open(row["path"]).convert("RGB"))[0]
    axes[0,col].imshow(dct, cmap="viridis"); axes[0,col].set_title("Real"); axes[0,col].axis("off")
for col, (_, row) in enumerate(sample_fake.iterrows()):
    dct = extract_dct_features(Image.open(row["path"]).convert("RGB"))[0]
    axes[1,col].imshow(dct, cmap="viridis"); axes[1,col].set_title("Fake"); axes[1,col].axis("off")
fig.suptitle("DCT Log-Magnitude Maps")
plt.tight_layout(); plt.show()

In [ ]:
class DCTDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        dct = extract_dct_features(Image.open(row["path"]).convert("RGB"))
        return torch.tensor(dct, dtype=torch.float), int(row["label"])

dct_train_dl = DataLoader(DCTDataset(split_dfs["train"]), batch_size=32, shuffle=True,  num_workers=2)
dct_val_dl   = DataLoader(DCTDataset(split_dfs["val"]),   batch_size=32, shuffle=False, num_workers=2)
dct_test_dl  = DataLoader(DCTDataset(split_dfs["test"]),  batch_size=32, shuffle=False, num_workers=2)

In [ ]:
# Matches backend/models/kyc/frequency_cnn.py exactly
class FrequencyCNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),          nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),         nn.ReLU(), nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*4*4, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )
    def forward(self, x): return self.classifier(self.features(x))

In [ ]:
freq_cnn  = FrequencyCNN().to(DEVICE)
opt_freq  = torch.optim.Adam(freq_cnn.parameters(), lr=1e-3)
FREQ_CKPT = WEIGHTS_DIR / "frequency_cnn_best.pt"

history_freq = train_model(
    freq_cnn, dct_train_dl, dct_val_dl,
    optimizer=opt_freq, scheduler=None,
    epochs=30, save_path=FREQ_CKPT, patience=7
)

In [ ]:
freq_cnn.load_state_dict(torch.load(FREQ_CKPT, map_location=DEVICE))
labels_freq, probs_freq = evaluate_loader(freq_cnn, dct_test_dl)
metrics_freq = evaluate(labels_freq, probs_freq, title="FrequencyCNN")

results_table = pd.concat([results_table, pd.DataFrame([{"Model": "FrequencyCNN",
    **{k: round(v,4) for k,v in metrics_freq.items()}}])], ignore_index=True)
results_table

---
## EXPERIMENT 5 — Model Complementarity

**Question:** Do the three models make different mistakes — justifying an ensemble?

In [ ]:
scores_df = pd.DataFrame({
    "label":         labels_eff,
    "efficientnet":  probs_eff,
    "vit":           probs_vit,
    "frequency_cnn": probs_freq,
})
print("Prediction correlation:")
print(scores_df[["efficientnet","vit","frequency_cnn"]].corr().round(4))

In [ ]:
preds_freq_bin = (probs_freq >= 0.5).astype(int)
n = len(preds_eff)
agree_all = ((preds_eff==preds_vit) & (preds_vit==preds_freq_bin)).sum()
agree_two = (
    ((preds_eff==preds_vit) & (preds_vit!=preds_freq_bin)) |
    ((preds_eff==preds_freq_bin) & (preds_eff!=preds_vit)) |
    ((preds_vit==preds_freq_bin) & (preds_vit!=preds_eff))
).sum()
print(f"All agree            : {agree_all} ({100*agree_all/n:.1f}%)")
print(f"Two agree / one out  : {agree_two} ({100*agree_two/n:.1f}%)")
print(f"All disagree         : {n-agree_all-agree_two} ({100*(n-agree_all-agree_two)/n:.1f}%)")

In [ ]:
err_eff  = set(np.where(preds_eff      != labels_eff)[0])
err_vit  = set(np.where(preds_vit      != labels_vit)[0])
err_freq = set(np.where(preds_freq_bin != labels_freq)[0])
print(f"EfficientNet errors     : {len(err_eff)}")
print(f"ViT errors              : {len(err_vit)}")
print(f"FrequencyCNN errors     : {len(err_freq)}")
print(f"Shared (all three)      : {len(err_eff & err_vit & err_freq)}")
print(f"Unique to EfficientNet  : {len(err_eff - err_vit - err_freq)}")
print(f"Unique to ViT           : {len(err_vit - err_eff - err_freq)}")
print(f"Unique to FrequencyCNN  : {len(err_freq - err_eff - err_vit)}")

---
## EXPERIMENT 6 — Ensemble

**Question:** Does combining all three models outperform each individual?

`ensemble_score = (efficientnet + vit + frequency_cnn) / 3`

In [ ]:
probs_ensemble = (probs_eff + probs_vit + probs_freq) / 3
metrics_ens = evaluate(labels_eff, probs_ensemble, title="Ensemble")
threshold_analysis(labels_eff, probs_ensemble, title="Ensemble — Threshold Analysis")

results_table = pd.concat([results_table, pd.DataFrame([{"Model": "Ensemble",
    **{k: round(v,4) for k,v in metrics_ens.items()}}])], ignore_index=True)
results_table

---
## 7. Production Risk-Tier Evaluation

| Score | Tier | Action |
|---|---|---|
| >= 0.75 | high_risk | reject |
| 0.50 - <0.75 | suspicious | manual_review |
| < 0.50 | verified | proceed |

In [ ]:
def assign_tier(score):
    if score >= 0.75: return "high_risk"
    if score >= 0.50: return "suspicious"
    return "verified"

tiers = [assign_tier(s) for s in probs_ensemble]
tier_df = pd.DataFrame({"tier": tiers, "label": labels_eff})
summary = tier_df.groupby("tier").agg(
    count=("label","count"),
    real=("label", lambda x: (x==0).sum()),
    fake=("label", lambda x: (x==1).sum()),
).reset_index()
print(summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(probs_ensemble[labels_eff==0], bins=50, alpha=0.6, label="Real", color="steelblue")
ax.hist(probs_ensemble[labels_eff==1], bins=50, alpha=0.6, label="Fake", color="crimson")
ax.axvline(0.50, color="orange", linestyle="--", label="suspicious (0.50)")
ax.axvline(0.75, color="red",    linestyle="--", label="high_risk (0.75)")
ax.set_xlabel("Ensemble Score"); ax.set_title("Score Distribution with Risk Tiers")
ax.legend(); plt.tight_layout(); plt.show()

---
## 8. Cross-Dataset Robustness

Evaluate each model separately on FaceForensics++ and Celeb-DF v2 test splits.

In [ ]:
def get_source_loader(source_name, dct=False, batch_size=32):
    mask = split_dfs["test"]["path"].str.contains(source_name)
    sub  = split_dfs["test"][mask]
    if len(sub) == 0: print(f"No test samples for {source_name}"); return None
    ds = DCTDataset(sub) if dct else FaceDataset(sub)
    return DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2)

robustness_rows = []
for source in ["faceforensics", "celebdf"]:
    rgb_dl = get_source_loader(source)
    dct_dl = get_source_loader(source, dct=True)
    if rgb_dl is None: continue
    y_true, p_e = evaluate_loader(effnet,    rgb_dl)
    _,       p_v = evaluate_loader(vit,       rgb_dl)
    _,       p_f = evaluate_loader(freq_cnn, dct_dl)
    p_ens = (p_e + p_v + p_f) / 3
    for name, probs in [("EfficientNet-B4",p_e),("ViT-B/16",p_v),
                         ("FrequencyCNN",p_f),("Ensemble",p_ens)]:
        robustness_rows.append({
            "Model": name, "Dataset": source,
            "ROC-AUC": round(roc_auc_score(y_true, probs), 4),
            "PR-AUC":  round(average_precision_score(y_true, probs), 4),
            "F1":      round(f1_score(y_true,(probs>=0.5).astype(int),zero_division=0), 4),
        })

robustness_df = pd.DataFrame(robustness_rows)
print(robustness_df.pivot(index="Model", columns="Dataset").to_string())

---
## 9. Error Analysis

In [ ]:
preds_ens = (probs_ensemble >= 0.5).astype(int)
fp_ens = np.where((labels_eff==0) & (preds_ens==1))[0]
fn_ens = np.where((labels_eff==1) & (preds_ens==0))[0]
print(f"Ensemble false positives: {len(fp_ens)}")
print(f"Ensemble false negatives: {len(fn_ens)}")
show_examples(fp_ens, probs_ensemble, "Ensemble FP — Real predicted as Fake")
show_examples(fn_ens, probs_ensemble, "Ensemble FN — Fake predicted as Real")

In [ ]:
# Show samples where EfficientNet and ViT strongly disagree
disagreement = np.abs(probs_eff - probs_vit)
top_idx = np.argsort(disagreement)[::-1][:6]

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
axes = axes.flatten()
for ax, i in zip(axes, top_idx):
    ax.imshow(Image.open(test_paths[i])); ax.axis("off")
    true_label = "Fake" if labels_eff[i] else "Real"
    ax.set_title(
        f"True={true_label}\nEff={probs_eff[i]:.2f} ViT={probs_vit[i]:.2f} Freq={probs_freq[i]:.2f}",
        fontsize=7)
fig.suptitle("Top Model Disagreements (EfficientNet vs ViT)")
plt.tight_layout(); plt.show()

---
## 10. Final Model Selection

Selection criteria (in order):
1. ROC-AUC / PR-AUC on test set
2. F1 at production threshold (0.50)
3. Cross-dataset robustness
4. Error complementarity
5. Ensemble improvement over individuals

In [ ]:
print("=== Final Comparison ===")
print(results_table.to_string(index=False))
print("\nSelected production system:")
print("  EfficientNet-B4 + ViT-B/16 + FrequencyCNN  ->  Simple Average Ensemble")

---
## 11. Backend Compatibility Check

In [ ]:
dummy = Image.fromarray(np.random.randint(0, 255, (224,224,3), dtype=np.uint8))

rgb_t = IMAGENET_TRANSFORM(dummy).unsqueeze(0).to(DEVICE)
assert rgb_t.shape == (1,3,224,224), "RGB shape mismatch"

dct_arr = extract_dct_features(dummy)
assert dct_arr.shape == (1,224,224), "DCT array shape mismatch"
dct_t = torch.tensor(dct_arr, dtype=torch.float).unsqueeze(0).to(DEVICE)
assert dct_t.shape == (1,1,224,224), "DCT tensor shape mismatch"

with torch.no_grad():
    p_e = float(F.softmax(effnet(rgb_t),   dim=1)[0,1])
    p_v = float(F.softmax(vit(rgb_t),      dim=1)[0,1])
    p_f = float(F.softmax(freq_cnn(dct_t), dim=1)[0,1])

ens = (p_e + p_v + p_f) / 3
print(f"EfficientNet : {p_e:.4f}")
print(f"ViT          : {p_v:.4f}")
print(f"FrequencyCNN : {p_f:.4f}")
print(f"Ensemble     : {ens:.4f}  [{assign_tier(ens)}]")
print("All compatibility checks passed.")

---
## 12. Generate Backend Artifacts

In [ ]:
for fname in ["efficientnet_best.pt", "vit_best.pt", "frequency_cnn_best.pt"]:
    p = WEIGHTS_DIR / fname
    print(f"  {fname:32s}: {'OK' if p.exists() else 'MISSING'}")

In [ ]:
def roc_to_list(y_true, probs, n=100):
    fpr, tpr, thr = roc_curve(y_true, probs)
    idx = np.linspace(0, len(fpr)-1, min(n, len(fpr)), dtype=int)
    return [{"fpr": round(float(fpr[i]),4), "tpr": round(float(tpr[i]),4),
             "threshold": round(float(thr[i]),4)} for i in idx]

kyc_metrics = {
    "models": {
        "efficientnet_b4": {**{k:round(v,4) for k,v in metrics_eff.items()},
                            "roc_curve": roc_to_list(labels_eff, probs_eff)},
        "vit_b16":         {**{k:round(v,4) for k,v in metrics_vit.items()},
                            "roc_curve": roc_to_list(labels_vit, probs_vit)},
        "frequency_cnn":   {**{k:round(v,4) for k,v in metrics_freq.items()},
                            "roc_curve": roc_to_list(labels_freq, probs_freq)},
        "ensemble":        {**{k:round(v,4) for k,v in metrics_ens.items()},
                            "roc_curve": roc_to_list(labels_eff, probs_ensemble)},
    },
    "datasets": [
        {"name": "FaceForensics++", "splits": {"train":0.8,"val":0.1,"test":0.1}},
        {"name": "Celeb-DF v2",     "splits": {"train":0.8,"val":0.1,"test":0.1}},
    ],
    "production_thresholds": {
        "verified":   {"lt": 0.50},
        "suspicious": {"gte": 0.50, "lt": 0.75},
        "high_risk":  {"gte": 0.75},
    },
    "preprocessing": {
        "max_frames":          KYC_MAX_VIDEO_FRAMES,
        "frame_size":          KYC_FRAME_SIZE,
        "face_detector":       "MTCNN",
        "rgb_normalization":   "ImageNet",
        "frequency_pipeline":  "grayscale -> 2D-DCT -> log1p(abs) -> [0,1]",
    },
    "robustness": robustness_df.to_dict(orient="records"),
}

out_path = PRECOMPUTED_DIR / "kyc_metrics.json"
with open(out_path, "w") as f:
    json.dump(kyc_metrics, f, indent=2)
print(f"Saved: {out_path}")

In [ ]:
print("\n=== Artifact Summary ===")
artifacts = [
    WEIGHTS_DIR / "efficientnet_best.pt",
    WEIGHTS_DIR / "vit_best.pt",
    WEIGHTS_DIR / "frequency_cnn_best.pt",
    PRECOMPUTED_DIR / "kyc_metrics.json",
]
for p in artifacts:
    size = f"{p.stat().st_size/1024:.0f} KB" if p.exists() else "MISSING"
    print(f"  {p.name:35s}: {size}")